# Stage 02 · Parallel Attestation Statistics

本 notebook 独立运行，只消费 stage 01 导出的 zip 包。


### 初始化运行常量与通用工具
定义 stage 02 的仓库路径、Drive 路径、脚本入口、缓存目录和基础工具函数，为后续读取 stage 01 包和执行统计流程做准备。
- 仓库：https://github.com/RICHAAARC/CEG-WM.git
- 分支：organize

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "02_Parallel_Attestation_Statistics"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
REPO_ROOT = Path("/content/ceg_wm_workspace")
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_Outputs_project_root"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
SCRIPT_PATH = REPO_ROOT / "scripts" / "02_Parallel_Attestation_Statistics.py"
ATTESTATION_ENV_PATH = DRIVE_PROJECT_ROOT / "secrets" / "attestation_env.json"
HF_HOME = REPO_ROOT / "huggingface_cache"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
SOURCE_PACKAGE_PATH = None

def run_checked(command, cwd=None, env=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path

def load_json(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))


### 挂载 Google Drive 并同步代码仓库
挂载 Google Drive、准备输出根目录，并克隆或更新代码仓库，确保后续并行 attestation 统计基于正确代码版本运行。

涉及的 Drive 根目录可以先按下面的树状结构理解：

```text
/content/drive/MyDrive/CEG_WM_Outputs_project_root/
├─ secrets/
│  └─ attestation_env.json 等运行密钥文件
├─ runtime_state/
│  ├─ 01_Paper_Full_Cuda/
│  │  └─ stage 01 各次运行的状态索引
│  └─ 02_Parallel_Attestation_Statistics/
│     └─ stage 02 各次运行的状态索引
├─ exports/
│  └─ 01_Paper_Full_Cuda/
│     └─ stage 01 导出的输入 zip 包
└─ logs/ 或各 stage 自带日志目录
   └─ 调试日志与命令执行日志
```

In [ ]:
from google.colab import drive


drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
ensure_dir(DRIVE_PROJECT_ROOT)
ensure_dir(HF_HOME)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_url = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    ).stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

print_json(
    "repo_binding",
    {
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "config_path": str(CONFIG_PATH),
        "script_path": str(SCRIPT_PATH),
    },
)


### 安装依赖并读取阶段角色配置
按 pyproject.toml 安装项目最小运行依赖，加载默认配置，并读取 parallel_attestation_statistics 段的关键参数，确认本 notebook 只消费 stage 01 导出的包而不重新执行主线生成流程。

stage 02 只消费 stage 01 的导出包并执行统计流程，无需主动下载基础模型或语义模型权重；仅在脚本真实运行路径要求时复用已有运行环境。

In [ ]:
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if Path("/usr/bin/apt-get").exists():
    subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
        check=False,
        capture_output=True,
    )

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)

if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

from scripts.notebook_runtime_common import load_yaml_mapping

CFG_OBJ = load_yaml_mapping(CONFIG_PATH)
PARALLEL_CFG = (
    CFG_OBJ.get("parallel_attestation_statistics")
    if isinstance(CFG_OBJ.get("parallel_attestation_statistics"), dict)
    else {}
)
ATTESTATION_ENV_STATUS = {
    "status": "not_requested",
    "reason": "stage_02_package_only_no_attestation_env_bootstrap",
}

print_json(
    "stage_role_binding",
    {
        "stage_name": NOTEBOOK_NAME,
        "mainline_reuse": "consume_stage_01_package_only",
        "calibration_score_name": PARALLEL_CFG.get("calibration_score_name"),
        "evaluate_score_name": PARALLEL_CFG.get("evaluate_score_name"),
        "model_download_policy": "skip_unless_runtime_path_requires_it",
    },
)
print_json("attestation_env_status", ATTESTATION_ENV_STATUS)

### 环境预检与 stage 01 输入包门禁
在执行统计脚本前，先检查配置文件、脚本文件、关键模块，以及 stage 01 导出包是否可发现。若存在硬性失败项，应先修复再继续执行。

In [ ]:
from scripts.notebook_runtime_common import resolve_stage_package_input_or_discover
from scripts.workflow_acceptance_common import detect_stage_02_preflight
import tempfile
import zipfile

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

EXPORT_STAGE_ROOT = DRIVE_PROJECT_ROOT / "exports" / "01_Paper_Full_Cuda"
PRECHECK_SOURCE_PACKAGE_RESOLUTION = resolve_stage_package_input_or_discover(
    SOURCE_PACKAGE_PATH,
    EXPORT_STAGE_ROOT,
    expected_stage_name="01_Paper_Full_Cuda",
)
PRECHECK_PACKAGE_CANDIDATES = PRECHECK_SOURCE_PACKAGE_RESOLUTION["candidates"]
PRECHECK_SELECTED_SOURCE_PACKAGE = (
    Path(str(PRECHECK_SOURCE_PACKAGE_RESOLUTION["selected_package_path"]))
    if PRECHECK_SOURCE_PACKAGE_RESOLUTION["selected_package_path"] != "<absent>"
    else None
)
PRECHECK_SOURCE_PACKAGE_PATH = PRECHECK_SELECTED_SOURCE_PACKAGE
RESOLVED_SOURCE_PACKAGE_PATH = PRECHECK_SELECTED_SOURCE_PACKAGE
RESOLVED_SOURCE_PACKAGE_INPUT = PRECHECK_SOURCE_PACKAGE_RESOLUTION.get("manual_input_path", "<absent>")
RESOLVED_SOURCE_PACKAGE_SOURCE = (
    "manual" if PRECHECK_SOURCE_PACKAGE_RESOLUTION.get("manual_input_used", False) else "auto_discovered"
)
PRECHECK_SOURCE_PACKAGE_SUMMARY = {
    "requested_package_input": RESOLVED_SOURCE_PACKAGE_INPUT,
    "resolved_package_path": str(RESOLVED_SOURCE_PACKAGE_PATH) if RESOLVED_SOURCE_PACKAGE_PATH is not None else "<absent>",
    "resolution_source": RESOLVED_SOURCE_PACKAGE_SOURCE,
    "selection_reason": PRECHECK_SOURCE_PACKAGE_RESOLUTION.get("selection_reason", "<absent>"),
    "auto_discovered_package_path": PRECHECK_SOURCE_PACKAGE_RESOLUTION.get("auto_discovered_package_path", "<absent>"),
    "resolution_error": PRECHECK_SOURCE_PACKAGE_RESOLUTION.get("resolution_error"),
}

PRECHECK_EXTRACTION_ROOT = Path(tempfile.mkdtemp(prefix="stage02_precheck_"))
PRECHECK_SOURCE_PACKAGE_FOR_PREFLIGHT = RESOLVED_SOURCE_PACKAGE_PATH or (PRECHECK_EXTRACTION_ROOT / "missing_source_package.zip")
PRECHECK_SOURCE_CONTRACT_PATH = PRECHECK_EXTRACTION_ROOT / "artifacts" / "parallel_attestation_statistics_input_contract.json"
PRECHECK_SOURCE_CONTRACT_DETAIL = "<absent>"
if RESOLVED_SOURCE_PACKAGE_PATH is not None and RESOLVED_SOURCE_PACKAGE_PATH.exists():
    try:
        with zipfile.ZipFile(RESOLVED_SOURCE_PACKAGE_PATH, "r") as archive_file:
            contract_member = "artifacts/parallel_attestation_statistics_input_contract.json"
            if contract_member in archive_file.namelist():
                archive_file.extract(contract_member, path=PRECHECK_EXTRACTION_ROOT)
                PRECHECK_SOURCE_CONTRACT_DETAIL = str(PRECHECK_SOURCE_CONTRACT_PATH)
            else:
                PRECHECK_SOURCE_CONTRACT_DETAIL = "contract member missing in source package"
    except Exception as exc:
        PRECHECK_SOURCE_CONTRACT_DETAIL = f"{type(exc).__name__}: {exc}"

STAGE_02_PREFLIGHT = detect_stage_02_preflight(
    CONFIG_PATH,
    PRECHECK_SOURCE_PACKAGE_FOR_PREFLIGHT,
    PRECHECK_SOURCE_CONTRACT_PATH,
)
record_precheck(
    "stage 02 preflight",
    STAGE_02_PREFLIGHT["ok"],
    json.dumps(STAGE_02_PREFLIGHT, ensure_ascii=False, sort_keys=True),
)
record_precheck(
    "GPU 工具可用（非硬要求）",
    STAGE_02_PREFLIGHT["gpu_tool_available"],
    STAGE_02_PREFLIGHT["nvidia_smi_path"],
)
record_precheck(
    "attestation env 状态（非硬要求）",
    not STAGE_02_PREFLIGHT["missing_attestation_env_vars"],
    ", ".join(STAGE_02_PREFLIGHT["missing_attestation_env_vars"]) or "ok",
)
record_precheck("配置文件存在", CONFIG_PATH.exists(), str(CONFIG_PATH))
record_precheck("主流程脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))

for module_name in ["yaml", "huggingface_hub"]:
    try:
        __import__(module_name)
        record_precheck(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        record_precheck(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

if PRECHECK_SOURCE_PACKAGE_RESOLUTION["manual_input_used"]:
    record_precheck(
        "手动输入包存在",
        PRECHECK_SOURCE_PACKAGE_RESOLUTION["package_exists"],
        str(RESOLVED_SOURCE_PACKAGE_PATH or PRECHECK_SOURCE_PACKAGE_RESOLUTION["manual_input_path"]),
    )
else:
    record_precheck(
        "stage 01 输入包可发现",
        PRECHECK_SOURCE_PACKAGE_RESOLUTION["selected_package_valid"],
        str(EXPORT_STAGE_ROOT),
    )
record_precheck(
    "stage 01 source package 解析",
    PRECHECK_SOURCE_PACKAGE_RESOLUTION["selected_package_valid"],
    json.dumps(PRECHECK_SOURCE_PACKAGE_SUMMARY, ensure_ascii=False, sort_keys=True),
)
record_precheck(
    "stage 01 source contract 可发现",
    PRECHECK_SOURCE_CONTRACT_PATH.exists(),
    PRECHECK_SOURCE_CONTRACT_DETAIL,
)

disk_usage = shutil.disk_usage(str(REPO_ROOT))
free_gb = disk_usage.free / (1024 ** 3)
record_precheck("磁盘剩余空间", free_gb >= 10.0, f"free={free_gb:.1f} GB，建议 >= 10 GB")

hard_fail = []
if not STAGE_02_PREFLIGHT["ok"]:
    hard_fail.append({
        "name": "stage 02 preflight",
        "detail": json.dumps(STAGE_02_PREFLIGHT, ensure_ascii=False, sort_keys=True),
    })

print_json(
    "environment_precheck",
    {
        "items": PRECHECK_RESULTS,
        "stage_02_preflight": STAGE_02_PREFLIGHT,
        "source_package_resolution": PRECHECK_SOURCE_PACKAGE_SUMMARY,
        "hard_fail": hard_fail,
    },
)

if hard_fail:
    raise RuntimeError(f"environment precheck failed: {hard_fail}")

### 选择 stage 01 输入包并执行统计脚本
自动发现或手动使用 stage 01 的导出 zip 包，选定输入后调用 scripts/02_Parallel_Attestation_Statistics.py，并读取本次 stage 的 summary 与 manifest。

检查“输入包位置”和“当前 stage 状态索引”两部分：

```text
DRIVE_PROJECT_ROOT/
├─ exports/
│  └─ 01_Paper_Full_Cuda/
│     └─ <stage_01_package>.zip
└─ runtime_state/
   └─ 02_Parallel_Attestation_Statistics/
      └─ <stage_run_id>/
         └─ stage_summary.json
            ├─ stage_manifest_path
            ├─ package_manifest_path
            ├─ run_root
            ├─ log_root
            └─ export_root
```

一端绑定 stage 01 的导出包，另一端把 stage 02 本次运行的所有正式输出路径统一登记到 stage_summary.json，供后续校验与诊断反查。

In [ ]:
from pathlib import Path

from scripts.notebook_runtime_common import build_repo_import_subprocess_env, make_stage_run_id

if "RESOLVED_SOURCE_PACKAGE_PATH" not in globals():
    raise RuntimeError("resolved source package is missing; run the precheck cell first")
if RESOLVED_SOURCE_PACKAGE_PATH is None:
    raise FileNotFoundError(
        PRECHECK_SOURCE_PACKAGE_RESOLUTION.get("resolution_error")
        or f"no valid stage 01 package resolved under {DRIVE_PROJECT_ROOT / 'exports' / '01_Paper_Full_Cuda'}"
    )

SOURCE_PACKAGE = Path(RESOLVED_SOURCE_PACKAGE_PATH)
PACKAGE_CANDIDATES = PRECHECK_PACKAGE_CANDIDATES
SELECTED_PACKAGE = dict(PRECHECK_SOURCE_PACKAGE_RESOLUTION.get("selected_candidate", {}))

if RESOLVED_SOURCE_PACKAGE_SOURCE == "auto_discovered":
    print_json(
        "package_candidates",
        [
            {
                "package_path": item.get("package_path"),
                "package_created_at": item.get("package_created_at"),
                "stage_run_id": item.get("stage_run_id"),
                "validation_error": item.get("validation_error"),
            }
            for item in PACKAGE_CANDIDATES
        ],
    )

print_json(
    "selected_package",
    {
        "requested_package_input": RESOLVED_SOURCE_PACKAGE_INPUT,
        "resolved_package_path": str(SOURCE_PACKAGE),
        "resolution_source": RESOLVED_SOURCE_PACKAGE_SOURCE,
        "selection_reason": PRECHECK_SOURCE_PACKAGE_RESOLUTION.get(
            "selection_reason",
            "manual SOURCE_PACKAGE_PATH override",
        ),
    },
)
STAGE_RUN_ID = make_stage_run_id(NOTEBOOK_NAME)
COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--config",
    str(CONFIG_PATH),
    "--source-package",
    str(SOURCE_PACKAGE),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--stage-run-id",
    STAGE_RUN_ID,
]
NOTEBOOK_SUBPROCESS_ENV = build_repo_import_subprocess_env(repo_root=REPO_ROOT)
run_checked(COMMAND, cwd=REPO_ROOT, env=NOTEBOOK_SUBPROCESS_ENV)

SUMMARY_PATH = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / STAGE_RUN_ID / "stage_summary.json"
STAGE_SUMMARY = load_json(SUMMARY_PATH)
STAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["stage_manifest_path"]))
PACKAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["package_manifest_path"]))
SOURCE_STAGE_MANIFEST = load_json(Path(STAGE_MANIFEST["source_stage_manifest_path"]))
SOURCE_PACKAGE_MANIFEST = load_json(Path(STAGE_MANIFEST["source_package_manifest_path"]))
print_json("stage_summary", STAGE_SUMMARY)
print_json("stage_manifest", STAGE_MANIFEST)
print_json("package_manifest", PACKAGE_MANIFEST)

### 校验输入包与当前阶段产物完整性
验证来源包的 sha256、一致性来源关系，以及当前 stage 02 产出的 calibration、evaluate、thresholds、report 和 workflow summary 是否齐全。

重点校验的是来源链条与 stage 02 自身产物目录是否完整，可按下面的树状结构理解：

```text
<run_root>/
├─ records/
│  ├─ calibration_record.json        [本阶段新增/重算]
│  └─ evaluate_record.json           [本阶段新增/重算]
├─ artifacts/
│  └─ thresholds_artifact.json       [本阶段新增]
├─ reports/ 或对应结果文件目录
│  ├─ evaluation_report.*            [本阶段新增]
│  └─ workflow_summary.*             [本阶段新增]
├─ stage_manifest.json 或 stage_manifest_path 指向的文件
├─ package_manifest.json 或 package_manifest_path 指向的文件
└─ source lineage
   ├─ source_package_path            [来自 stage 01]
   ├─ source_stage_manifest_path     [来自 stage 01]
   └─ source_package_manifest_path   [来自 stage 01]
```

要确认的不只是文件存在，还包括：输入包哈希一致、source_stage_name 确实指向 stage 01，以及本阶段新增记录和报告都已经完整落盘。

In [ ]:
from scripts.notebook_runtime_common import collect_missing_file_entries, compute_file_sha256

SOURCE_PACKAGE_ACTUAL_PATH = Path(STAGE_MANIFEST["source_package_path"])
SOURCE_PACKAGE_ACTUAL_SHA256 = compute_file_sha256(SOURCE_PACKAGE_ACTUAL_PATH)
if SOURCE_PACKAGE_ACTUAL_SHA256 != STAGE_MANIFEST["source_package_sha256"]:
    raise RuntimeError(
        f"source package sha256 mismatch: manifest={STAGE_MANIFEST['source_package_sha256']} actual={SOURCE_PACKAGE_ACTUAL_SHA256}"
    )
if SOURCE_STAGE_MANIFEST.get("stage_name") != "01_Paper_Full_Cuda":
    raise RuntimeError(f"unexpected source stage_name: {SOURCE_STAGE_MANIFEST.get('stage_name')}")
REQUIRED_STAGE_FILES = {
    "stage_manifest": Path(STAGE_SUMMARY["stage_manifest_path"]),
    "package_manifest": Path(STAGE_SUMMARY["package_manifest_path"]),
    "source_package": SOURCE_PACKAGE_ACTUAL_PATH,
    "source_stage_manifest": Path(STAGE_MANIFEST["source_stage_manifest_path"]),
    "source_package_manifest": Path(STAGE_MANIFEST["source_package_manifest_path"]),
    "calibration_record": Path(STAGE_MANIFEST["records"]["calibration_record"]["path"]),
    "evaluate_record": Path(STAGE_MANIFEST["records"]["evaluate_record"]["path"]),
    "thresholds_artifact": Path(STAGE_MANIFEST["thresholds_path"]),
    "evaluation_report": Path(STAGE_MANIFEST["evaluation_report_path"]),
    "workflow_summary": Path(STAGE_MANIFEST["workflow_summary_path"]),
}
MISSING_STAGE_FILES = collect_missing_file_entries(REQUIRED_STAGE_FILES)
if MISSING_STAGE_FILES:
    raise FileNotFoundError(f"stage 02 validation failed, missing files: {MISSING_STAGE_FILES}")
VALIDATION_RESULT = {
    "stage_name": NOTEBOOK_NAME,
    "stage_run_id": STAGE_RUN_ID,
    "source_stage_run_id": STAGE_MANIFEST["source_stage_run_id"],
    "source_package_sha256": SOURCE_PACKAGE_ACTUAL_SHA256,
    "missing_files": MISSING_STAGE_FILES,
    "status": "ok",
}
print_json("validation_result", VALIDATION_RESULT)


### 汇总 lineage 与日志诊断信息
汇总 source package、source stage、当前 manifest 关键字段以及最新日志内容，帮助快速定位 stage 02 的输入来源和运行问题。

诊断时建议优先按下面的目录关系查看：

```text
stage_summary.json
├─ source_package_path     -> stage 01 导出的 zip 包
├─ run_root                -> 当前 stage 02 实际产物目录
├─ log_root                -> 当前 stage 02 日志目录
├─ runtime_state_root      -> 当前 notebook 的状态索引目录
└─ export_root             -> 当前 stage 02 导出目录
<log_root>/
└─ *.log
<run_root>/
├─ records/
├─ artifacts/
└─ reports/ 或对应结果文件目录
```

如果前面步骤失败，优先看 log_root；如果要确认 stage 02 到底新生成了什么，优先看 run_root 和 export_root。

In [ ]:
from scripts.notebook_runtime_common import summarize_manifest_fields, tail_text_file

LOG_ROOT = Path(STAGE_SUMMARY["log_root"])
LOG_FILES = sorted(LOG_ROOT.rglob("*.log"), key=lambda item: item.stat().st_mtime if item.exists() else 0)
LATEST_LOG_PATH = LOG_FILES[-1] if LOG_FILES else None
DIAGNOSTIC_RESULT = {
    "stage_run_id": STAGE_RUN_ID,
    "source_stage_run_id": STAGE_MANIFEST["source_stage_run_id"],
    "source_package_path": STAGE_MANIFEST["source_package_path"],
    "source_package_sha256": STAGE_MANIFEST["source_package_sha256"],
    "missing_files": VALIDATION_RESULT["missing_files"],
    "latest_log_path": str(LATEST_LOG_PATH) if LATEST_LOG_PATH else "<absent>",
    "latest_log_tail": tail_text_file(LATEST_LOG_PATH, max_lines=20) if LATEST_LOG_PATH else [],
    "source_lineage_summary": summarize_manifest_fields(
        STAGE_MANIFEST,
        [
            "source_stage_name",
            "source_stage_run_id",
            "source_package_manifest_path",
            "source_package_manifest_digest",
            "source_runtime_config_snapshot_path",
            "source_prompt_snapshot_path",
        ],
    ),
    "current_manifest_summary": summarize_manifest_fields(
        STAGE_MANIFEST,
        [
            "stage_name",
            "stage_run_id",
            "thresholds_path",
            "workflow_summary_path",
        ],
    ),
    "status": "ok" if not VALIDATION_RESULT["missing_files"] else "diagnostic_required",
}
print_json("diagnostic_result", DIAGNOSTIC_RESULT)


### 最终保存在 Google Drive 中的文件路径
本 notebook 的输入和输出都位于 Google Drive 挂载目录下。下面沿用 stage 01 第 14 个单元的展示方式，并额外标出哪些目录或文件是本阶段新增的。

```text
/content/drive/MyDrive/CEG_WM_Outputs_project_root/
├─ runtime_state/
│  ├─ 01_Paper_Full_Cuda/
│  │  └─ <source_stage_run_id>/
│  │     └─ stage_summary.json        [来自 stage 01]
│  └─ 02_Parallel_Attestation_Statistics/
│     └─ <stage_run_id>/
│        └─ stage_summary.json        [本阶段新增]
├─ exports/
│  ├─ 01_Paper_Full_Cuda/
│  │  └─ <stage_01_package>.zip       [来自 stage 01，作为本阶段输入]
│  └─ 02_Parallel_Attestation_Statistics/
│     └─ <stage_02_package>.zip       [本阶段新增]
├─ <run_root，对应 stage 02 stage_summary.json 中的 run_root>/
│  ├─ records/
│  │  ├─ calibration_record.json      [本阶段新增/重算]
│  │  └─ evaluate_record.json         [本阶段新增/重算]
│  ├─ artifacts/
│  │  ├─ thresholds/
│  │  │  ├─ thresholds_artifact.json  [本阶段新增]
│  │  │  └─ threshold_metadata_artifact.json [本阶段新增]
│  │  ├─ evaluation_report.json       [本阶段新增]
│  │  ├─ workflow_summary.json        [本阶段新增]
│  │  ├─ run_closure.json             [本阶段新增]
│  │  └─ stage_manifest.json          [本阶段新增]
│  └─ 其他本阶段运行目录
├─ <export_root，对应 stage 02 stage_summary.json 中的 export_root>/
│  ├─ 02_Parallel_Attestation_Statistics__<stage_run_id>__from__<source_stage_run_id>.zip
│  ├─ package_manifest.json
│  └─ package_index.json
├─ <package_path，对应 stage 02 stage_summary.json 中的 package_path>
│  └─ stage 02 最终 zip 包             [本阶段新增]
└─ <log_root，对应 stage 02 stage_summary.json 中的 log_root>/
   └─ *.log                           [本阶段新增]
```

最终 zip 包以 stage 02 自己的产物为主，不会把 stage 01 的完整 records 或完整 zip 再打进去；它只保留 stage 01 的 lineage 快照。zip 内部的核心相对路径如下：

```text
records/calibration_record.json
records/evaluate_record.json
artifacts/thresholds/thresholds_artifact.json
artifacts/thresholds/threshold_metadata_artifact.json
artifacts/evaluation_report.json
artifacts/run_closure.json
artifacts/workflow_summary.json
artifacts/stage_manifest.json
artifacts/package_manifest.json
artifacts/package_index.json
runtime_metadata/runtime_config_snapshot.yaml
lineage/source_stage_manifest.json
lineage/source_package_manifest.json
```

其中：
1. 本阶段新增结果是 calibration_record、evaluate_record、thresholds、evaluation_report、workflow_summary、run_closure。
2. 来自 stage 01 的只是一层 lineage 信息，即 source_stage_manifest.json 和 source_package_manifest.json。
3. stage 02 zip 不包含 stage 01 的 embed_record、detect_record、calibration_record、evaluate_record。

建议的查找顺序是：
1. 先打开 stage 02 的 stage_summary.json。
2. 根据其中的 run_root、log_root、export_root、package_path 跳转到真实落盘位置。
3. 如果要核对输入来源，查看 exports/01_Paper_Full_Cuda 下的 source package，以及 runtime_state/01_Paper_Full_Cuda 下对应 source_stage_run_id 的 stage_summary.json。
4. 如果只关心本阶段新增结果，优先查看 run_root、export_root、package_path 和 log_root。